## What is a Runnable?
Runnables are the core building blocks of the LangChain Expression Language LCEL. A Runnable is a standardized unit of work that allows different components like ChatOpenAI, prompts, and tools to be composed into unified pipelines using the pipe (|) operator.
### Interface Requirements
To be a valid Runnable, an object must inherit from the base Runnable class and implement a specific set of synchronous and asynchronous standard methods. This strict interface contract ensures that all components can seamlessly chain into each other.
### Core Synchronous Methods

* invoke(input, config): Transforms a single input into an output.
* batch(inputs, config): Processes multiple inputs in parallel.
* stream(input, config): Streams out data chunks as they are generated.

### Core Asynchronous Methods

* ainvoke(input, config): Async version of invoke.
* abatch(inputs, config): Async version of batch.
* astream(input, config): Async version of stream.

### The 6 Main Types of Runnables
LangChain separates runnables into task-specific classes (like models and prompts) and orchestration primitives. The 6 foundational orchestration primitives include

#### 1. RunnableSequence
Invokes a series of runnables sequentially. The output of one step automatically serves as the input to the next. This is implicitly created when using the | operator. [2, 6, 9] 

In [10]:
from langchain_core.runnables import RunnableLambda

# Chain two basic transformations sequentially
step1 = RunnableLambda(lambda x: x + 2)
step2 = RunnableLambda(lambda x: x * 3)

chain = step1 | step2
print(chain.invoke(1))  # Output: (1 + 2) * 3 = 9

9


#### 2. RunnableParallel
Executes multiple runnables concurrently using the same input. It outputs a dictionary matching the defined keys. [1, 2, 8, 9, 10] 

In [11]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

step_a = RunnableLambda(lambda x: x.upper())
step_b = RunnableLambda(lambda x: len(x))

# Run both operations on the same string simultaneously
parallel_chain = RunnableParallel(uppercase=step_a, length=step_b)
print(parallel_chain.invoke("hello"))  # Output: {'uppercase': 'HELLO', 'length': 5}

{'uppercase': 'HELLO', 'length': 5}


#### 3. RunnableLambda
Converts any arbitrary Python custom function into a full-fledged Runnable. This allows regular code to benefit from batching, async execution, and LangSmith tracing.

In [12]:
from langchain_core.runnables import RunnableLambda


def custom_format(text: str) -> str:
    return f"***{text.strip()}***"


runnable_func = RunnableLambda(custom_format)
print(runnable_func.invoke("  alert  "))  # Output: ***alert***

***alert***


#### 4. RunnablePassthrough
Passes the input directly through unchanged. It is primarily used to forward data to later steps in a sequence or dynamically inject extra fields into a dictionary pipeline.

In [13]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)

# Retain original query while generating an alternative format
pipeline = RunnableParallel(
    original=RunnablePassthrough(), modified=RunnableLambda(lambda x: x.lower())
)
print(pipeline.invoke("INPUT"))  # Output: {'original': 'INPUT', 'modified': 'input'}

{'original': 'INPUT', 'modified': 'input'}


#### 5. RunnableBranch
Implements conditional routing. It evaluates a series of (condition, runnable) pairs and executes the runnable corresponding to the first true condition.

In [14]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

is_short = RunnableLambda(lambda x: len(x) < 5)
handle_short = RunnableLambda(lambda x: "Short string")
handle_long = RunnableLambda(lambda x: "Long string")

# Route data conditionally based on string length
branch = RunnableBranch((is_short, handle_short), handle_long)  # Default branch
print(branch.invoke("abc"))  # Output: Short string
print(branch.invoke("long_text"))  # Output: Long string

Short string
Long string


#### 6. RunnableBinding
Wraps an existing Runnable to attach structural metadata, configurations, error fallbacks, or target keywords without changing its core functional execution block.

In [ ]:
from langchain_core.runnables import RunnableLambda

base_runnable = RunnableLambda(lambda x: x * 2)

# Bind structural configuration details to create a new Runnable
configured_runnable = base_runnable.with_config(tags=["math-operation"])
print(configured_runnable.invoke(5))  # Output: 10

10
